# Parallax Validation Walkthrough

This notebook provides an interactive, step-by-step walkthrough of the Parallax validation process. You will see exactly how we load data, detect communities (Attention Heads), and perform multi-hop reasoning with Community-Structured Attention (CSA).

## Objectives:
1. **Visualize the Graph**: See the topological structure of the canonical `toy_graph`.
2. **DSCF in Action**: Observe how Local (LPA) and Global (Modularity) signals fuse to form communities.
3. **Reasoning Journey**: Follow a multi-hop traversal and see the attention scores at every step.
4. **The Bridge Bonus (EF-005)**: Demonstration of how we solve the 'Type Alignment Trap'.

In [ ]:
import os
import sys
from pathlib import Path
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np

# Add project root to path
sys.path.insert(0, str(Path(os.getcwd()).parent))

from adapters.csv_adapter import load_csv_adapter
from core.community_engine import dscf_communities, modularity_score
from core.embedding_engine import RandomEngine
from core.attention_engine import CSAEngine
from core.structural_encoder import build_community_distance_matrix, adjacent_community_pairs
from reasoning.traversal import BeamTraversal
from reasoning.answer_extractor import extract

print("Environment Ready.")

## 1. Load the Graph Environment

We use the `toy_graph.csv` which contains two primary conceptual clusters: **History** (Roman/Greek) and **Science** (Modern Physics).

In [ ]:
csv_path = Path("../tests/fixtures/toy_graph.csv")
adapter = load_csv_adapter(str(csv_path))
G = adapter.to_networkx()

print(f"Loaded graph with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges.")

plt.figure(figsize=(10, 8))
pos = nx.spring_layout(G, seed=42)
nx.draw(G, pos, with_labels=True, node_color='lightblue', edge_color='gray', node_size=2000, font_size=10)
plt.title("Canonical Toy Graph Topology")
plt.show()

## 2. DSCF: Forming Attention Heads

We now run the **Dual-Signal Community Fusion** algorithm. Unlike standard methods, DSCF ensures that every community is both locally tight and globally significant.

In [ ]:
# Run DSCF
np.random.seed(42)
parts = dscf_communities(G, resolution=1.0)
q = modularity_score(G, parts)
print(f"Detected {len(parts)} communities with Modularity Q = {q:.4f}")

# Map nodes to colors based on community
community_map = {node: i for i, members in enumerate(parts) for node in members}
colors = [community_map[node] for node in G.nodes()]

plt.figure(figsize=(10, 8))
nx.draw(G, pos, with_labels=True, node_color=colors, cmap=plt.cm.Set3, node_size=2000, font_size=10)
plt.title("Visualizing Attention Heads (DSCF Communities)")
plt.show()

## 3. The Reasoning Journey

We will now ask Parallax: **"What is connected to 'newton'?"**

We'll use the **Metaedge Bridge Bonus (EF-005)** to favor the `INFLUENCED` relationship, simulating a user interested in intellectual lineage.

In [ ]:
# Setup Engines
embed_engine = RandomEngine(dim=64)
embeddings = embed_engine.encode_entities({n: n for n in G.nodes()})

dist_matrix = build_community_distance_matrix(G, community_map)
adj_pairs = adjacent_community_pairs(G, community_map)
csa_engine = CSAEngine(communities=community_map, embeddings=embeddings)
csa_engine.set_community_graph(dist_matrix, adj_pairs)

# Configure Traversal with the Bridge Bonus
edge_weights = {"INFLUENCED": 0.4}

traversal = BeamTraversal(
    adapter=adapter,
    csa_engine=csa_engine,
    embeddings=embeddings,
    communities=community_map,
    beam_width=10,
    max_hop=3,
    edge_type_weights=edge_weights
)

paths = traversal.traverse(["newton"])
answers = extract(paths, top_k=5)

print("Top reasoning paths found:")
for i, ans in enumerate(answers):
    print(f"\n[{i+1}] {ans.entity_id} (Score: {ans.score:.4f})")
    print(f"    Path: {' -> '.join(ans.best_path.nodes)}")
    print(f"    Heads: {ans.community_trace}")

## 4. Deep Dive: Attention Weights

Let's look under the hood at how the **CSA Formula** scored the `newton -> einstein` edge compared to a cross-domain jump.

In [ ]:
w_internal = csa_engine.compute_weight("newton", "einstein", hop=1, edge_type="INFLUENCED", edge_type_weights=edge_weights)
w_external = csa_engine.compute_weight("newton", "caesar", hop=1, edge_type="NONE", edge_type_weights=edge_weights)

print(f"Attention (Internal: Newton -> Einstein): {w_internal:.4f}")
print(f"Attention (External: Newton -> Caesar):   {w_external:.4f}")
print(f"\nStructural guidance steered the beam {w_internal/w_external:.1f}x more strongly toward the relevant domain.")

## Conclusion

This walkthrough demonstrates that Parallax doesn't just 'find' data—it **reasons** through it using structural attention. Every step is verifiable, and every path follows the conceptual boundaries of the graph.